## Intro to Data Preprocessing

In [8]:
import pandas as pd

# Load data
volunteer = pd.read_csv('data/volunteer.csv')

In [9]:
# How many missing values are in the locality column?
volunteer['locality'].isnull().sum()

np.int64(70)

In [10]:
# Drop the Latitude and Longitude columns from volunteer
volunteer_cols = volunteer.drop(['Latitude', 'Longitude'], axis=1)

# Drop rows with missing category_desc values from volunteer_cols
volunteer_subset = volunteer_cols.dropna(subset=['category_desc'])

# Print out the shape of the subset
print(volunteer_subset.shape)

(617, 33)


In [11]:
# Which datatypes are in the volunteer
print(volunteer_subset.dtypes)

opportunity_id          int64
content_id              int64
vol_requests            int64
event_time              int64
title                     str
hits                    int64
summary                   str
is_priority               str
category_id           float64
category_desc             str
amsl                  float64
amsl_unit             float64
org_title                 str
org_content_id          int64
addresses_count         int64
locality                  str
region                    str
postalcode            float64
primary_loc           float64
display_url               str
recurrence_type           str
hours                   int64
created_date              str
last_modified_date        str
start_date_date           str
end_date_date             str
status                    str
Community Board       float64
Community Council     float64
Census Tract          float64
BIN                   float64
BBL                   float64
NTA                   float64
dtype: obj

In [12]:
# Print the head of the hits column
print(volunteer["hits"].head())

0    737
1     22
2     62
3     14
4     31
Name: hits, dtype: int64


In [14]:

# Convert the hits column to type float
volunteer["hits"] = volunteer["hits"].astype(float)

# Look at the dtypes of the dataset
print(volunteer.dtypes)

opportunity_id          int64
content_id              int64
vol_requests            int64
event_time              int64
title                     str
hits                  float64
summary                   str
is_priority               str
category_id           float64
category_desc             str
amsl                  float64
amsl_unit             float64
org_title                 str
org_content_id          int64
addresses_count         int64
locality                  str
region                    str
postalcode            float64
primary_loc           float64
display_url               str
recurrence_type           str
hours                   int64
created_date              str
last_modified_date        str
start_date_date           str
end_date_date             str
status                    str
Latitude              float64
Longitude             float64
Community Board       float64
Community Council     float64
Census Tract          float64
BIN                   float64
BBL       

In [19]:
# Class imbalance

# How many times does each description occur in the volunteer dataset?
print(volunteer['category_desc'].value_counts(), end='\n\n')

# Which descriptions occur less than 50 times in the volunteer dataset?
print(volunteer['category_desc'].value_counts() < 50, end='\n\n')

# Print names of descriptions that occur less than 50 times in the volunteer dataset
print(volunteer['category_desc'].value_counts()[volunteer['category_desc'].value_counts() < 50].index.tolist())

category_desc
Strengthening Communities    307
Helping Neighbors in Need    119
Education                     92
Health                        52
Environment                   32
Emergency Preparedness        15
Name: count, dtype: int64

category_desc
Strengthening Communities    False
Helping Neighbors in Need    False
Education                    False
Health                       False
Environment                   True
Emergency Preparedness        True
Name: count, dtype: bool

['Environment', 'Emergency Preparedness']


In [23]:
# Drop missing values from the category_desc column
volunteer = volunteer.dropna(subset=['category_desc'])

In [25]:
# Stratified sampling
from sklearn.model_selection import train_test_split

# Create a DataFrame with all columns except category_desc
X = volunteer.drop(columns=['category_desc'])

# Create a category_desc labels dataset
y = volunteer[['category_desc']]

# Use stratified sampling to split up the dataset according to the y dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Print the category_desc counts from y_train
print(y_train['category_desc'].value_counts())

category_desc
Strengthening Communities    230
Helping Neighbors in Need     89
Education                     69
Health                        39
Environment                   24
Emergency Preparedness        11
Name: count, dtype: int64


## Standardization

In [39]:
# Load wine dataset
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X = wine.data
y = wine.target

wine = wine.frame

### Modeling without normalizing

In [40]:
from sklearn.neighbors import KNeighborsClassifier

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

knn = KNeighborsClassifier()

# Fit the knn model to the training data
knn.fit(X_train, y_train)

# Score the model on the test data
print(knn.score(X_test, y_test))

0.7777777777777778


### Log Normalization

In [41]:
# Check the variance of the columns in the wine dataset
print(wine.var())

alcohol                             0.659062
malic_acid                          1.248015
ash                                 0.075265
alcalinity_of_ash                  11.152686
magnesium                         203.989335
total_phenols                       0.391690
flavanoids                          0.997719
nonflavanoid_phenols                0.015489
proanthocyanins                     0.327595
color_intensity                     5.374449
hue                                 0.052245
od280/od315_of_diluted_wines        0.504086
proline                         99166.717355
target                              0.600679
dtype: float64


In [42]:
import numpy as np
# Print out the variance of the Proline column
print(wine['proline'].var())

# Apply the log normalization function to the Proline column
wine['Proline_log'] = np.log(wine['proline'])

# Check the variance of the normalized Proline column
print(wine['Proline_log'].var())

99166.71735542436
0.17231366191842012


### Scaling 

In [45]:
# You want to use the Ash, Alcalinity of ash, and Magnesium columns in the wine dataset to train a linear model,
# but it's possible that these columns are all measured in different ways, which would bias a linear model.

print(wine[['ash', 'alcalinity_of_ash', 'magnesium', 'alcohol', 'malic_acid']].describe())

              ash  alcalinity_of_ash   magnesium     alcohol  malic_acid
count  178.000000         178.000000  178.000000  178.000000  178.000000
mean     2.366517          19.494944   99.741573   13.000618    2.336348
std      0.274344           3.339564   14.282484    0.811827    1.117146
min      1.360000          10.600000   70.000000   11.030000    0.740000
25%      2.210000          17.200000   88.000000   12.362500    1.602500
50%      2.360000          19.500000   98.000000   13.050000    1.865000
75%      2.557500          21.500000  107.000000   13.677500    3.082500
max      3.230000          30.000000  162.000000   14.830000    5.800000


In [53]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Subset the DataFrame you want to scale 
wine_subset = wine[['ash', 'alcalinity_of_ash', 'magnesium']]

# Apply the scaler to wine_subset
wine_subset_scaled = scaler.fit_transform(wine_subset)

# Check the mean of the scaled data
print(wine_subset_scaled.std(axis=0))

[1. 1. 1.]


In [54]:
# KNN on non-scaled data
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

# Split the dataset and labels into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Fit the k-nearest neighbors model to the training data
knn.fit(X_train, y_train)

# Score the model on the test data
print(knn.score(X_test, y_test))

0.7777777777777778


In [55]:
# KNN on scaled data

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Instantiate a StandardScaler
scaler = StandardScaler()

# Scale the training and test features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the k-nearest neighbors model to the training data
knn.fit(X_train_scaled, y_train)

# Score the model on the test data
print(knn.score(X_test_scaled, y_test))

0.9333333333333333


## Feature Engineering

### Encoding categorical variables

In [59]:
# Load hiking dataset from json
hiking = pd.read_json('data/hiking.json')

In [61]:
from sklearn.preprocessing import LabelEncoder

# Set up the LabelEncoder object
enc = LabelEncoder()

# Apply the encoding to the "Accessible" column
hiking['Accessible_enc'] = enc.fit_transform(hiking['Accessible'])

# Compare the two columns
print(hiking[['Accessible', 'Accessible_enc']].head())

  Accessible  Accessible_enc
0          Y               1
1          N               0
2          N               0
3          N               0
4          N               0


In [63]:
# Transform the category_desc column
category_enc = pd.get_dummies(volunteer['category_desc'])

# Take a look at the encoded columns
print(category_enc.head())

   Education  Emergency Preparedness  Environment  Health  \
1      False                   False        False   False   
2      False                   False        False   False   
3      False                   False        False   False   
4      False                   False         True   False   
5      False                   False         True   False   

   Helping Neighbors in Need  Strengthening Communities  
1                      False                       True  
2                      False                       True  
3                      False                       True  
4                      False                      False  
5                      False                      False  


### Aggregating numerical features

In [66]:
running_times_5k = pd.read_csv('data/running_times_5k.csv', index_col=0)

print(running_times_5k.head())

    name  run1  run2  run3  run4  run5
0    Sue  20.1  18.5  19.6  20.3  18.3
1   Mark  16.5  17.1  16.9  17.6  17.3
2   Sean  23.5  25.1  25.2  24.6  23.9
3   Erin  21.7  21.1  20.9  22.1  22.2
4  Jenny  25.8  27.1  26.1  26.7  26.9


In [67]:
# Use .loc to create a mean column
running_times_5k["mean"] = running_times_5k.loc[:, 'run1':'run5'].mean(axis=1)

# Take a look at the results
print(running_times_5k.head())

    name  run1  run2  run3  run4  run5   mean
0    Sue  20.1  18.5  19.6  20.3  18.3  19.36
1   Mark  16.5  17.1  16.9  17.6  17.3  17.08
2   Sean  23.5  25.1  25.2  24.6  23.9  24.46
3   Erin  21.7  21.1  20.9  22.1  22.2  21.60
4  Jenny  25.8  27.1  26.1  26.7  26.9  26.52


In [69]:
print(volunteer.head())

   opportunity_id  content_id  vol_requests  event_time  \
1            5008       37036             2           0   
2            5016       37143            20           0   
3            5022       37237           500           0   
4            5055       37425            15           0   
5            5056       37426            15           0   

                                               title   hits  \
1                                       Web designer   22.0   
2      Urban Adventures - Ice Skating at Lasker Rink   62.0   
3  Fight global hunger and support women farmers ...   14.0   
4                                      Stop 'N' Swap   31.0   
5                               Queens Stop 'N' Swap  135.0   

                                             summary is_priority  category_id  \
1             Build a website for an Afghan business         NaN          1.0   
2  Please join us and the students from Mott Hall...         NaN          1.0   
3  The Oxfam Action Cor

In [70]:
# Extracting datetime columns

# First, convert string column to date column
volunteer["start_date_converted"] = pd.to_datetime(volunteer["start_date_date"])

# Extract just the month from the converted column
volunteer["start_date_month"] = volunteer["start_date_converted"].dt.month

# Take a look at the converted and new month columns
print(volunteer[[ "start_date_converted", "start_date_month" ]].head())

  start_date_converted  start_date_month
1           2011-02-01                 2
2           2011-01-29                 1
3           2011-02-14                 2
4           2011-02-05                 2
5           2011-02-12                 2


### Engineering text features

In [86]:
hiking = pd.read_json('data/hiking.json')

hiking = hiking.dropna(subset=['Length'])

In [87]:
import re

# Write a pattern to extract numbers and decimals
def return_mileage(length):
    
    # Search the text for matches
    mile = re.search(r"(\d+\.\d+)", length)
    
    # If a value is returned, use group(0) to return the found value
    if mile is not None:
        return float(mile.group(0))
        
# Apply the function to the Length column and take a look at both columns
hiking["Length_num"] = hiking["Length"].apply(return_mileage)
print(hiking[["Length", "Length_num"]].head())

       Length  Length_num
0   0.8 miles        0.80
1    1.0 mile        1.00
2  0.75 miles        0.75
3   0.5 miles        0.50
4   0.5 miles        0.50


In [91]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorizing text data
volunteer = pd.read_csv('data/volunteer.csv')

volunteer = volunteer.dropna(subset=['category_desc'])

# Take the title text
title_text = volunteer['title'].values

# Create the vectorizer method
tfidf_vec = TfidfVectorizer()

# Transform the text into tf-idf vectors
text_tfidf = tfidf_vec.fit_transform(title_text)

In [93]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB as NaiveBayes

# Split the dataset according to the class distribution of category_desc
y = volunteer["category_desc"].dropna()
X_train, X_test, y_train, y_test = train_test_split(text_tfidf.toarray(), y, stratify=y, random_state=42)

nb = NaiveBayes()

# Fit the model to the training data
nb.fit(X_train, y_train)

# Print out the model's accuracy
print(nb.score(X_test, y_test))

0.5548387096774193


## Selecting Features for Modeling

In [96]:
volunteer = pd.read_csv('data/volunteer2.csv', index_col=0)
volunteer.head()

,vol_requests,title,hits,category_desc,locality,region,postalcode,created_date,vol_requests_lognorm,created_month,Education,Emergency Preparedness,Environment,Health,Helping Neighbors in Need,Strengthening Communities
1,2,Web designer,22,Strengthening Communities,"5 22nd St\nNew York, NY 10010\n(40.74053152272...",NY,10010.0,2011-01-14,0.693147,1,False,False,False,False,False,True
2,20,Urban Adventures - Ice Skating at Lasker Rink,62,Strengthening Communities,NaN,NY,10026.0,2011-01-19,2.995732,1,False,False,False,False,False,True
3,500,Fight global hunger and support women farmers ...,14,Strengthening Communities,NaN,NY,2114.0,2011-01-21,6.214608,1,False,False,False,False,False,True
4,15,Stop 'N' Swap,31,Environment,NaN,NY,10455.0,2011-01-28,2.708050,1,False,False,True,False,False,False
5,15,Queens Stop 'N' Swap,135,Environment,NaN,NY,11372.0,2011-01-28,2.708050,1,False,False,True,False,False,False


In [99]:
# Create a list of redundant column names to drop
to_drop = ["category_desc", "locality", "region", "vol_requests", "created_date"]

# Drop those columns from the dataset
volunteer_subset = volunteer.drop(to_drop, axis=1)

# Print out the head of volunteer_subset
volunteer_subset.head()

,title,hits,postalcode,vol_requests_lognorm,created_month,Education,Emergency Preparedness,Environment,Health,Helping Neighbors in Need,Strengthening Communities
1,Web designer,22,10010.0,0.693147,1,False,False,False,False,False,True
2,Urban Adventures - Ice Skating at Lasker Rink,62,10026.0,2.995732,1,False,False,False,False,False,True
3,Fight global hunger and support women farmers ...,14,2114.0,6.214608,1,False,False,False,False,False,True
4,Stop 'N' Swap,31,10455.0,2.708050,1,False,False,True,False,False,False
5,Queens Stop 'N' Swap,135,11372.0,2.708050,1,False,False,True,False,False,False


In [109]:
# Check correlated features
wine = pd.read_csv('data/wine.csv', index_col=0)

# Print out the column correlations of the wine dataset
wine.corr()

,Flavanoids,Total phenols,Malic acid,OD280/OD315 of diluted wines,Hue
Flavanoids,1.000000,0.864564,-0.411007,0.787194,0.543479
Total phenols,0.864564,1.000000,-0.335167,0.699949,0.433681
Malic acid,-0.411007,-0.335167,1.000000,-0.368710,-0.561296
OD280/OD315 of diluted wines,0.787194,0.699949,-0.368710,1.000000,0.565468
Hue,0.543479,0.433681,-0.561296,0.565468,1.000000


In [110]:
# Drop the column with a correlation of approximately 0.8 with another column
wine = wine.drop('Flavanoids', axis=1)

wine.head()

,Total phenols,Malic acid,OD280/OD315 of diluted wines,Hue
0,2.80,1.71,3.92,1.04
1,2.65,1.78,3.40,1.05
2,2.80,2.36,3.17,1.03
3,3.85,1.95,3.45,0.86
4,2.80,2.59,2.93,1.04


### Selecting features using text vectors

In [119]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorizing text data
volunteer = pd.read_csv('data/volunteer.csv')

volunteer = volunteer.dropna(subset=['category_desc'])

# Take the title text
title_text = volunteer['title'].values

# Create the vectorizer method
tfidf_vec = TfidfVectorizer()

# Transform the text into tf-idf vectors
text_tfidf = tfidf_vec.fit_transform(title_text)

In [120]:
def return_weights(vocab, original_vocab, vector, vector_index, top_n):
    zipped = dict(zip(vector[vector_index].indices, vector[vector_index].data))
    
    # Transform that zipped dict into a series
    zipped_series = pd.Series({vocab[i]:zipped[i] for i in vector[vector_index].indices})
    
    # Sort the series to pull out the top n weighted words
    zipped_index = zipped_series.sort_values(ascending=False)[:top_n].index
    return [original_vocab[i] for i in zipped_index]

# Print out the weighted words
print(return_weights(tfidf_vec.get_feature_names_out(), tfidf_vec.vocabulary_, text_tfidf, 8, 3))

[190, 946, 468]


In [121]:
def words_to_filter(vocab, original_vocab, vector, top_n):
    filter_list = []
    for i in range(0, vector.shape[0]):
    
        # Call the return_weights function and extend filter_list
        filtered = return_weights(vocab, original_vocab, vector, i, top_n)
        filter_list.extend(filtered)
        
    # Return the list in a set, so we don't get duplicate word indices
    return set(filter_list)

# Call the function to get the list of word indices
filtered_words = words_to_filter(tfidf_vec.get_feature_names_out(), tfidf_vec.vocabulary_, text_tfidf, 3)

# Filter the columns in text_tfidf to only those in filtered_words
filtered_text = text_tfidf[:, list(filtered_words)]

In [123]:
# Training Naive Bayes with feature selection
y = volunteer["category_desc"]
nb = NaiveBayes()

# Split the dataset according to the class distribution of category_desc
X_train, X_test, y_train, y_test = train_test_split(filtered_text.toarray(), y, stratify=y, random_state=42)

# Fit the model to the training data
nb.fit(X_train, y_train)

# Print out the model's accuracy
print(nb.score(X_test, y_test))

0.5548387096774193


### Dimensionality reduction

In [132]:
# Instantiate a PCA object
from sklearn.decomposition import PCA

pca = PCA()
wine = load_wine(as_frame=True)
wine = wine.frame

# Change target column name to "Type"
wine = wine.rename(columns={"target": "Type"})

In [133]:
# Define the features and labels from the wine dataset
X = wine.drop("Type", axis=1)
y = wine["Type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Apply PCA to the wine dataset X vector
pca_X_train = pca.fit_transform(X_train)
pca_X_test = pca.transform(X_test)

# Look at the percentage of variance explained by the different components
print(pca.explained_variance_ratio_)

[9.97795009e-01 2.02071827e-03 9.88350594e-05 5.66222566e-05
 1.26161135e-05 8.93235789e-06 3.13856866e-06 1.57406401e-06
 1.15918860e-06 7.49332354e-07 3.70332305e-07 1.94185373e-07
 8.08440051e-08]


In [134]:
# Fit knn to the training data
knn = KNeighborsClassifier()
knn.fit(pca_X_train, y_train)

# Score knn on the test data and print it out
print(knn.score(pca_X_test, y_test))

0.7777777777777778


## Putting It All Together

In [199]:
# Load UFO sightings dataset
ufo = pd.read_csv('data/ufo.csv')

print(ufo.info())

<class 'pandas.DataFrame'>
RangeIndex: 4935 entries, 0 to 4934
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4935 non-null   str    
 1   city            4926 non-null   str    
 2   state           4516 non-null   str    
 3   country         4255 non-null   str    
 4   type            4776 non-null   str    
 5   seconds         4935 non-null   float64
 6   length_of_time  4792 non-null   str    
 7   desc            4932 non-null   str    
 8   recorded        4935 non-null   str    
 9   lat             4935 non-null   str    
 10  long            4935 non-null   float64
dtypes: float64(2), str(9)
memory usage: 424.2 KB
None


In [200]:
ufo['date'] = pd.to_datetime(ufo['date'])

In [201]:
# Count the missing values in the length_of_time, state, and type columns, in that order
print(ufo[['length_of_time', 'state', 'type']].isnull().sum())

# Drop rows where length_of_time, state, or type are missing
ufo_no_missing = ufo.dropna(subset=['length_of_time', 'state', 'type'])

# Print out the shape of the new dataset
print(ufo_no_missing.shape)

length_of_time    143
state             419
type              159
dtype: int64
(4283, 11)


In [202]:
ufo = ufo_no_missing
ufo['length_of_time']

0               2 weeks
1                30sec.
3       about 5 minutes
4                     2
5            10 minutes
             ...       
4930    about 5 seconds
4931         25 seconds
4932      early morning
4933            2 hours
4934          1 minutes
Name: length_of_time, Length: 4283, dtype: str

In [203]:
def return_minutes(time_string):

    # Search for numbers in time_string
    num = re.search(r'\d+', str(time_string))
    if num is not None:
        return int(num.group(0))
        
# Apply the extraction to the length_of_time column
ufo["minutes"] = ufo["length_of_time"].apply(return_minutes)

# Take a look at the head of both of the columns
print(ufo[['length_of_time', 'minutes']].head())

    length_of_time  minutes
0          2 weeks      2.0
1           30sec.     30.0
3  about 5 minutes      5.0
4                2      2.0
5       10 minutes     10.0


In [204]:
# Check the variance of the seconds and minutes columns
print(ufo[['seconds', 'minutes']].var())

# Log normalize the seconds column
ufo["seconds_log"] = ufo["seconds"].apply(lambda x: np.log(x) if x > 0 else 0)

# Print out the variance of just the seconds_log column
print(ufo[['seconds_log']].var())

seconds    1.545212e+10
minutes    9.173689e+02
dtype: float64
seconds_log    5.677784
dtype: float64


In [205]:
# Use pandas to encode us values as 1 and others as 0
ufo["country_enc"] = ufo["country"].map({"us" : 1}).fillna(0).astype(int)

# Print the number of unique type values
print(len(ufo['type'].unique()))

# Create a one-hot encoded set of the type values
type_set = pd.get_dummies(ufo['type'])

# Concatenate this set back to the ufo DataFrame
ufo = pd.concat([ufo, type_set], axis=1)

21


In [206]:
# Look at the first 5 rows of the date column
print(ufo["date"].head())

# Extract the month from the date column
ufo["month"] = ufo["date"].dt.month

# Extract the year from the date column
ufo["year"] = ufo["date"].dt.year

# Take a look at the head of all three columns
print(ufo[["date", "month", "year"]].head())

0   2011-11-03 19:21:00
1   2004-10-03 19:05:00
3   2002-11-21 05:45:00
4   2010-08-19 12:55:00
5   2012-06-16 23:00:00
Name: date, dtype: datetime64[us]
                 date  month  year
0 2011-11-03 19:21:00     11  2011
1 2004-10-03 19:05:00     10  2004
3 2002-11-21 05:45:00     11  2002
4 2010-08-19 12:55:00      8  2010
5 2012-06-16 23:00:00      6  2012


In [207]:
# Take a look at the head of the desc field
print(ufo["desc"].head())

# Instantiate the tfidf vectorizer object
vec = TfidfVectorizer()

# Fit and transform desc using vec
desc_tfidf = vec.fit_transform(ufo["desc"])

# Look at the number of columns and rows
print(desc_tfidf.shape)

0    Red blinking objects similar to airplanes or s...
1                 Many fighter jets flying towards UFO
3    It was a large&#44 triangular shaped flying ob...
4       A white spinning disc in the shape of an oval.
5    Dancing lights that would fly around and then ...
Name: desc, dtype: str
(4283, 5754)


In [208]:
# Now to get rid of some of the unnecessary features in the ufo dataset. 
# Because the country column has been encoded as country_enc,
# you can select it and drop the other columns related to location: city, country, lat, long, and state.
to_drop = ['country', 'city', 'lat', 'long', 'state', 'length_of_time', 'desc', 'seconds', 'minutes', 'date', 'recorded']

# Drop those features
ufo_dropped = ufo.drop(columns=to_drop)

# Let's also filter some words out of the text vector we created

# Use the words_to_filter() function you created previously; pass in vocab, vec.vocabulary_,
# desc_tfidf, and keep the top 4 words as the last parameter.
filtered_words = words_to_filter(vec.get_feature_names_out(), vec.vocabulary_, desc_tfidf, 4)

In [210]:
# Take a look at the features in the X set of data
X = ufo_dropped.drop(['type', 'country_enc'], axis=1)

# The y labels are the encoded country column, where 1 is "us" and 0 is "ca".
y = ufo['country'].map({"us" : 1, "ca" : 0}).fillna(0).astype(int)

print(X.columns)

# Split the X and y sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Fit knn to the training sets
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

# Print the score of knn on the test sets
print(knn.score(X_test, y_test))

Index(['seconds_log', 'changing', 'chevron', 'cigar', 'circle', 'cone',
       'cross', 'cylinder', 'diamond', 'disk', 'egg', 'fireball', 'flash',
       'formation', 'light', 'other', 'oval', 'rectangle', 'sphere',
       'teardrop', 'triangle', 'unknown', 'month', 'year'],
      dtype='str')
0.8487394957983193


In [212]:
# Use the list of filtered words we created to filter the text vector
filtered_text = desc_tfidf[:, list(filtered_words)]

# Split the X and y sets using train_test_split, setting stratify=y 
X_train, X_test, y_train, y_test = train_test_split(filtered_text.toarray(), y, stratify=y, random_state=42)

# Fit nb to the training sets
nb = NaiveBayes()
nb.fit(X_train, y_train)

# Print the score of nb on the test sets
print(nb.score(X_test, y_test))

0.861811391223156
